# Install Dependecies

In [11]:
%%capture
%pip install -q "nltk>=3.9,<4" "spacy>=3.8,<4" "transformers>=5,<6"
%pip install matplotlib
!python -m spacy download en_core_web_sm
!python -m spacy download fr_core_news_sm
%pip install ipynbname
%pip install datasets
%pip install ipywidgets

In [12]:
%%capture
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
%pip install ipykernel

# Imports

In [13]:
import ipynbname
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import re
import time
from collections import Counter
from datasets import load_dataset
from pathlib import Path
from tqdm.std import tqdm
from transformers import AutoTokenizer

# Import datasets

In [5]:
dataset = load_dataset("coastalcph/tydi_xor_rc")
df_train = dataset["train"].to_pandas()
df_val = dataset["validation"].to_pandas()

# Preprocessing

In [7]:
LANGUAGES = ["ar", "ko", "te"]

In [8]:
# select languages
df_train = df_train[df_train["lang"].isin(LANGUAGES)].copy()
df_val = df_val[df_val["lang"].isin(LANGUAGES)].copy()

# 3 Week 37: Structured Span Prediction
* Convert the character-level answer offsets into BIO labels over context tokens. 
* Add automatic checks for at least the following cases: 
  - an answer at character 0, 
  - a multi-token answer, 
  - punctuation adjacent to an answer and an unanswerable example. 
* Document how subword pieces are handled if applicable. 
* Implement one question-conditioned sequence labeller for the group: the representation of the question must influence the predicted label for every context token. 
* Compare it with a simple span baseline. An empty-output baseline is sufficient. If you use lexical overlap, select context tokens using only overlap with the question (optionally after fixed preprocessing or translation), convert the best contiguous run to a span and never use gold answer text or offsets. The correct output for an unanswerable question is an empty span. Evaluate and analyse the models according to Section 1.


In [43]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]", re.UNICODE)

In [44]:
# lab_2.ipynb
def tokenize_with_offsets(text):
    matches = list(TOKEN_PATTERN.finditer(text))
    tokens = [match.group(0) for match in matches]
    offsets = [(match.start(), match.end()) for match in matches]
    return tokens, offsets

In [45]:
# lab_2.ipynb
def character_span_to_bio(context, answer_start=None, answer_text=""):
    tokens, offsets = tokenize_with_offsets(context)
    labels = ["O"] * len(tokens)

    if answer_start is None or answer_text == "":
        return tokens, offsets, labels

    answer_end = answer_start + len(answer_text)
    if context[answer_start:answer_end] != answer_text:
        raise ValueError("The supplied answer text does not match the character span")

    covered = [
        index
        for index, (start, end) in enumerate(offsets)
        if start < answer_end and end > answer_start
    ]
    if not covered:
        raise ValueError("The answer does not overlap any token")
    if offsets[covered[0]][0] != answer_start or offsets[covered[-1]][1] != answer_end:
        raise ValueError("The answer span does not align with token boundaries")

    labels[covered[0]] = "B-ANS"
    for index in covered[1:]:
        labels[index] = "I-ANS"
    return tokens, offsets, labels

In [19]:
# lab_2.ipynb
def bio_to_character_span(context, offsets, labels):
    if len(offsets) != len(labels):
        raise ValueError("Offsets and labels must have the same length")
    if not set(labels) <= {"O", "B-ANS", "I-ANS"}:
        raise ValueError("Only O, B-ANS and I-ANS labels are supported")

    answer_indices = [
        index for index, label in enumerate(labels) if label != "O"
    ]
    if not answer_indices:
        return None, ""

    start_index = answer_indices[0]
    expected_indices = list(range(start_index, start_index + len(answer_indices)))
    expected_labels = ["B-ANS"] + ["I-ANS"] * (len(answer_indices) - 1)
    if answer_indices != expected_indices:
        raise ValueError("The labels contain multiple or non-contiguous answer spans")
    if [labels[index] for index in answer_indices] != expected_labels:
        raise ValueError("The answer span must begin with B-ANS and continue with I-ANS")

    start = offsets[answer_indices[0]][0]
    end = offsets[answer_indices[-1]][1]
    return start, context[start:end]

### Automatic checks

In [20]:
# lab_2.ipynb: answer at character 0 and multi-token answer
context = "Ada Lovelace wrote the first algorithm."
answer_text = "Ada Lovelace"
answer_start = context.index(answer_text)

tokens, offsets, labels = character_span_to_bio(
    context, answer_start, answer_text
)
round_trip_start, round_trip_text = bio_to_character_span(
    context, offsets, labels
)

assert answer_start == 0
assert round_trip_start == answer_start
assert round_trip_text == answer_text
assert labels[:2] == ["B-ANS", "I-ANS"]

pd.DataFrame({"token": tokens, "offset": offsets, "label": labels})

,token,offset,label
0,Ada,"(0, 3)",B-ANS
1,Lovelace,"(4, 12)",I-ANS
2,wrote,"(13, 18)",O
3,the,"(19, 22)",O
4,first,"(23, 28)",O
5,algorithm,"(29, 38)",O
6,.,"(38, 39)",O


In [ ]:
tokenize_with_offsets

In [22]:
# claude: punctuation adjacent to the answer
third_context = "Paris, the capital of France, lies on the Seine."
for third_answer in ["Paris", "France", "the Seine"]:
    third_start = third_context.index(third_answer)
    third_tokens, third_offsets, third_labels = character_span_to_bio(
        third_context, third_start, third_answer
    )
    assert bio_to_character_span(
        third_context, third_offsets, third_labels
    ) == (third_start, third_answer)
    # the neighbouring punctuation is its own token and stays outside the span
    answer_tokens = [token for token, label in zip(third_tokens, third_labels) if label != "O"]
    assert " ".join(answer_tokens) == third_answer

pd.DataFrame({"token": third_tokens, "offset": third_offsets, "label": third_labels})

,token,offset,label
0,Paris,"(0, 5)",O
1,",","(5, 6)",O
2,the,"(7, 10)",O
3,capital,"(11, 18)",O
4,of,"(19, 21)",O
5,France,"(22, 28)",O
6,",","(28, 29)",O
7,lies,"(30, 34)",O
8,on,"(35, 37)",O
9,the,"(38, 41)",B-ANS


In [23]:
# claude: a span that cuts through a word is rejected
fourth_context = "It is an ancient Egyptian symbol."
try:
    character_span_to_bio(fourth_context, fourth_context.index("Egypt"), "Egypt")
    raise AssertionError("A span inside a word should be rejected")
except ValueError as error:
    print("Rejected as expected:", error)

Rejected as expected: The answer span does not align with token boundaries


### Apply the conversion to project data

In [25]:
# claude
def project_answer(answerable, answer_start, answer):
    # unanswerable examples are stored with answer_start == -1 and a placeholder answer
    if not answerable:
        return None, ""
    return int(answer_start), answer


def convert_split(df):
    rows = []
    for context, answerable, answer_start, answer in zip(
        df["context"], df["answerable"], df["answer_start"], df["answer"]
    ):
        answer_start, answer_text = project_answer(answerable, answer_start, answer)
        try:
            tokens, offsets, labels = character_span_to_bio(context, answer_start, answer_text)
            assert bio_to_character_span(context, offsets, labels) == (answer_start, answer_text)
            error = None
        except ValueError as exception:
            tokens, offsets, labels, error = None, None, None, str(exception)
        rows.append({"tokens": tokens, "offsets": offsets, "labels": labels, "error": error})
    return pd.DataFrame(rows, index=df.index)

In [26]:
bio_train = convert_split(df_train)
bio_val = convert_split(df_val)

In [27]:
# claude
def conversion_report(df, bio):
    report = pd.DataFrame({
        "n_checked": df.groupby("lang").size(),
        "n_answerable": df.groupby("lang")["answerable"].sum(),
        "n_unanswerable": (~df["answerable"]).groupby(df["lang"]).sum(),
        "n_failed": bio["error"].notna().groupby(df["lang"]).sum(),
    })
    report.loc["all"] = report.sum()
    return report

In [28]:
# train
display(conversion_report(df_train, bio_train))
print(bio_train["error"].value_counts())

,n_checked,n_answerable,n_unanswerable,n_failed
lang,,,,
ar,2558,2303,255,23
ko,2422,2359,63,45
te,1355,1310,45,11
all,6335,5972,363,79


error
The answer span does not align with token boundaries    79
Name: count, dtype: int64


In [29]:
# val
display(conversion_report(df_val, bio_val))
print(bio_val["error"].value_counts())

,n_checked,n_answerable,n_unanswerable,n_failed
lang,,,,
ar,415,363,52,0
ko,356,337,19,4
te,384,291,93,1
all,1155,991,164,5


error
The answer span does not align with token boundaries    5
Name: count, dtype: int64


In [30]:
# claude
def show_failures(df, bio, n=10, window=20):
    failed = df[bio["error"].notna()]
    return pd.DataFrame({
        "lang": failed["lang"],
        "answer": failed["answer"],
        "context_window": [
            context[max(0, start - window):start + len(answer) + window]
            for context, start, answer in zip(failed["context"], failed["answer_start"], failed["answer"])
        ],
    }).head(n)

In [31]:
# train
show_failures(df_train, bio_train)

,lang,answer,context_window
4861,ko,Dr John O'Sullivan with his colleagues Terence...,an radio-astronomer Dr John O'Sullivan with hi...
4873,ko,Egypt,Horus is an ancient Egyptian symbol of protec
4899,ko,social clas,"ough separate from, social class.\nThe biggest..."
4913,ko,The Battle of Leyte Gul,"The Battle of Leyte Gulf (Filipino: ""Labana"
4920,ko,eremy Bentham,"sentient animals. Jeremy Bentham, the founder..."
4950,ko,"r 17 countries in Asia and the Pacific, 9 coun...",ary energy source for 17 countries in Asia and...
4991,ko,the night of September 1,"enses at Incheon on the night of September 15,..."
5018,ko,several thousand smaller lake,nds. There are also several thousand smaller l...
5116,ko,he Korean Intellectual Property Office (KIPO) ...,The Korean Intellectual Property Office (KIPO)...
5130,ko,Lina Marcela Medina de Jurad,Lina Marcela Medina de Jurado (; born 23 Septemb


In [32]:
# val
show_failures(df_val, bio_val)

,lang,answer,context_window
131,te,IT Delhi,"Internationally, IIT Delhi was ranked 172 in t"
358,ko,Egypt,ave derived from an Egyptian hieroglyph depic
558,ko,"Keith Boak, Euros Lyn, Joe Ahearne, Brian Gran...","e role of producer. Keith Boak, Euros Lyn, Joe..."
773,ko,Ancient structures with possibly astronomical ...,ntified celestial oFAncient structures with po...
1279,ko,Egypt,"ve been based on an Egyptian hieroglyph F1 ,"


### Handling misaligned gold spans

In [33]:
# claude
def snap_answer_to_tokens(context, answer_start, answer_text):
    _, offsets = tokenize_with_offsets(context)
    answer_end = answer_start + len(answer_text)
    covered = [
        (start, end) for start, end in offsets if start < answer_end and end > answer_start
    ]
    start, end = covered[0][0], covered[-1][1]
    return start, context[start:end]


def snap_split(df, bio):
    df = df.copy()
    df["snapped"] = bio["error"].notna()
    for index in df.index[df["snapped"]]:
        df.loc[index, ["answer_start", "answer"]] = snap_answer_to_tokens(
            df.at[index, "context"], df.at[index, "answer_start"], df.at[index, "answer"]
        )
    return df

In [34]:
df_train = snap_split(df_train, bio_train)
df_val = snap_split(df_val, bio_val)

bio_train = convert_split(df_train)
bio_val = convert_split(df_val)
assert bio_train["error"].isna().all() and bio_val["error"].isna().all()

df_train[["tokens", "offsets", "labels"]] = bio_train[["tokens", "offsets", "labels"]]
df_val[["tokens", "offsets", "labels"]] = bio_val[["tokens", "offsets", "labels"]]

print("snapped train:", df_train.groupby("lang")["snapped"].sum().to_dict())
print("snapped val:", df_val.groupby("lang")["snapped"].sum().to_dict())

snapped train: {'ar': 23, 'ko': 45, 'te': 11}
snapped val: {'ar': 0, 'ko': 4, 'te': 1}


In [35]:
# week 36.3 check repeated on the snapped spans
for name, df in [("train", df_train), ("validation", df_val)]:
    ans = df[df["answerable"]]
    ok = [context[start:start + len(answer)] == answer
          for context, start, answer in zip(ans["context"], ans["answer_start"], ans["answer"])]
    print(f"{name}: answer: {len(ok)} - fail: {ok.count(False)}")

train: answer: 5972 - fail: 0
validation: answer: 991 - fail: 0


### Coverage of the required cases in the project data

In [36]:
# claude
PUNCTUATION = re.compile(r"[^\w\s]")

In [37]:
def case_table(df):
    answerable = df["answerable"]
    answer_end = df["answer_start"] + df["answer"].str.len()
    before = [context[start - 1] if start > 0 else "" for context, start in zip(df["context"], df["answer_start"])]
    after = [context[end] if end < len(context) else "" for context, end in zip(df["context"], answer_end)]
    n_answer_tokens = df["labels"].map(lambda labels: labels.count("B-ANS") + labels.count("I-ANS"))
    cases = pd.DataFrame({
        "lang": df["lang"],
        "answer at character 0": answerable & (df["answer_start"] == 0),
        "multi-token answer": answerable & (n_answer_tokens > 1),
        "punctuation adjacent": answerable & pd.Series(
            [bool(PUNCTUATION.match(b)) or bool(PUNCTUATION.match(a)) for b, a in zip(before, after)],
            index=df.index,
        ),
        "unanswerable (empty span)": ~answerable & (n_answer_tokens == 0),
    })
    table = cases.groupby("lang").sum()
    table.loc["all"] = table.sum()
    return table

In [38]:
# train
case_table(df_train)

,answer at character 0,multi-token answer,punctuation adjacent,unanswerable (empty span)
lang,,,,
ar,299,1527,1167,255
ko,303,1477,1123,63
te,196,740,521,45
all,798,3744,2811,363


In [39]:
# val
case_table(df_val)

,answer at character 0,multi-token answer,punctuation adjacent,unanswerable (empty span)
lang,,,,
ar,40,227,183,52
ko,41,206,156,19
te,61,153,112,93
all,142,586,451,164


In [40]:
# claude
def show_examples(df, n=5):
    return pd.DataFrame({
        "lang": df["lang"],
        "question": df["question"],
        "answer": df["answer"],
        "answer_start": df["answer_start"],
        "labelled tokens": [
            [(token, label) for token, label in zip(tokens, labels) if label != "O"]
            for tokens, labels in zip(df["tokens"], df["labels"])
        ],
        "n_context_tokens": df["tokens"].map(len),
    }).head(n)

In [41]:
# five answerable validation examples
show_examples(df_val[df_val["answerable"]])

,lang,question,answer,answer_start,labelled tokens,n_context_tokens
0,te,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland,0,"[(Portland, B-ANS)]",152
1,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Indian subcontinent,99,"[(Indian, B-ANS), (subcontinent, I-ANS)]",134
2,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,England,451,"[(England, B-ANS)]",138
3,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,1914,26,"[(1914, B-ANS)]",215
4,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,28 July 1914,155,"[(28, B-ANS), (July, I-ANS), (1914, I-ANS)]",129


In [42]:
# five unanswerable validation examples (empty span, all labels O)
show_examples(df_val[~df_val["answerable"]])

,lang,question,answer,answer_start,labelled tokens,n_context_tokens
364,ko,시차는 중력과 관련이 있는가?,no,-1,[],73
368,ko,맹장 없이 살 수 있을까?,no,-1,[],383
369,ko,무한은 공식이 있을까?,no,-1,[],50
398,ko,핵융합은 상용화 가능성이 있는가?,no,-1,[],135
417,ko,화성의 대기에 인간이 살 수 있는가?,no,-1,[],221
